In [4]:
import argparse
import ast
import html
import json
import re
import ssl
import urllib.request

PAGE_URL = "https://pswsnetwork.eng.ua.edu/home"
BASE_URL = "https://pswsnetwork.eng.ua.edu/"

# Notebook / default settings (toggle these directly when running in Jupyter)
INCLUDE_RETIRED = False
VERIFY_SSL = False
DEFAULT_OUTPUT = "psws_stations_openspace.geojson"

# Maps each JS variable on the page to (status label, hex color, OpenSpace point-size)
GROUPS = {
    "geojson_online": ("Online", "#008000", 6.0),
    "geojson_poss": ("Possibly Offline", "#FFA500", 6.0),
    "geojson_offline": ("Offline", "#FF0000", 6.0),
    "geojson_retired": ("Retired", "#808080", 5.0),
}


def fetch_psws_geojson(
    url: str = PAGE_URL,
    include_retired: bool = True,
    verify_ssl: bool = False,
) -> dict:
    ssl_context = ssl.create_default_context()
    if not verify_ssl:
        ssl_context.check_hostname = False
        ssl_context.verify_mode = ssl.CERT_NONE

    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; GeoJSONFetcher/1.0)"},
    )
    with urllib.request.urlopen(
        req, timeout=30, context=ssl_context
    ) as response:
        raw_html = response.read().decode("utf-8")

    features = []
    for var_name, (status, color, point_size) in GROUPS.items():
        if status == "Retired" and not include_retired:
            continue

        match = re.search(rf"var\s+{var_name}\s*=\s*(\{{.*?\}});", raw_html, re.S)
        if not match:
            continue

        # Quote unquoted JS object keys so ast.literal_eval can parse the literal
        js_obj = re.sub(
            r"([{,]\s*)(type|features|geometry|coordinates|properties|title|URL|ID)\s*:",
            r'\1"\2":',
            match.group(1),
        )
        collection = ast.literal_eval(js_obj)

        for feat in collection["features"]:
            props = feat["properties"]
            clean_title = html.unescape(props["title"]).strip()
            station_id = str(props["ID"]).strip()

            # OpenSpace native styling & UI label properties
            props["name"] = f"{clean_title} ({station_id})"
            props["color"] = color
            props["point-size"] = point_size

            # Preserved station metadata
            props["title"] = clean_title
            props["full_url"] = f"{BASE_URL}{props['URL']}{station_id}"
            props["status"] = status

            features.append(feat)

    return {
        "type": "FeatureCollection",
        "features": features,
    }


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Fetch PSWS station locations as an OpenSpace-ready GeoJSON."
    )
    parser.add_argument(
        "-o",
        "--output",
        default=DEFAULT_OUTPUT,
        help=f"Output GeoJSON path (default: {DEFAULT_OUTPUT})",
    )
    parser.add_argument(
        "--no-retired",
        action="store_true",
        help="Exclude retired stations from the generated GeoJSON.",
    )
    parser.add_argument(
        "--verify-ssl",
        action="store_true",
        default=VERIFY_SSL,
        help="Enable strict SSL certificate verification.",
    )
    # parse_known_args ignores Jupyter/IPython's -f kernel arguments
    args, _ = parser.parse_known_args()

    include_retired = INCLUDE_RETIRED and not args.no_retired
    geojson_data = fetch_psws_geojson(
        include_retired=include_retired,
        verify_ssl=args.verify_ssl,
    )

    with open(args.output, "w", encoding="utf-8") as f:
        json.dump(geojson_data, f, indent=2)

    print(f"Wrote {len(geojson_data['features'])} features to {args.output}")


if __name__ == "__main__":
    main()

Wrote 90 features to psws_stations_openspace.geojson
